<a href="https://colab.research.google.com/github/268aishwarya-alt/Drug-Synergy/blob/main/PS_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#phase 1
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import tensorflow as tf
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os

In [2]:
#phase 2[data collection and preprocessing]
print("\nCleaning scored combinations..");
drugcombs = pd.read_csv("/content/drugcombs_scored.csv");
print(drugcombs.isnull().sum())
drugcombs = drugcombs.dropna(subset=['Loewe'])
print("Duplicates = ",drugcombs.duplicated().sum())
drugcombs = drugcombs.dropna(subset=['Drug1', 'Drug2'])
print(drugcombs.isnull().sum())




Cleaning scored combinations..
ID           0
Drug1        2
Drug2        8
Cell line    0
ZIP          0
Bliss        0
Loewe        1
HSA          0
dtype: int64
Duplicates =  0
ID           0
Drug1        0
Drug2        0
Cell line    0
ZIP          0
Bliss        0
Loewe        0
HSA          0
dtype: int64


In [3]:
print("\nCleaning chemical information..")
chem_info = pd.read_csv("/content/drug_chemical_info.csv", encoding = 'latin1')
chem_info['drugNameOfficial'] = chem_info['drugNameOfficial'].fillna(chem_info['drugName'])
#dictionary mapping its drug name to its molecular weigths
chem_dict = dict(zip(chem_info['drugNameOfficial'], chem_info['molecularWeight']))
print(chem_dict)
drugcombs['Drug1_MW'] = drugcombs['Drug1'].map(chem_dict).fillna(0.0)
drugcombs['Drug2_MW'] = drugcombs['Drug2'].map(chem_dict).fillna(0.0)



Cleaning chemical information..
{'bendamustine': 358.26284, 'lonidamine': 321.1581, 'lenalidomide': 259.26062, 'cladribine': 285.68698, 'pentostatin': 268.26914, '6-mercaptopurine': 152.17706, 'everolimus': 958.22442, 'aminoglutethimide': 232.27834, '17-dione': 302.40794, 'letrozole': 285.30274, 'Pemetrexed disodium': 0.0, 'Gemcitabine': 0.0, 'Tamoxifen': 0.0, 'vincristine sulfate': 923.03612, 'combretastatin A-4': 316.3484, 'plinabulin': 336.3877, 'Docetaxel': 0.0, 'blebbistatin': 292.33184, 'Ispinesib': 0.0, 'doxorubicin': 543.51926, 'topotecan': 457.90676, 'cyclophosphamide': 261.085962, '5-azacytidine': 244.20468, 'Lyovac Cosmegen': 1255.41704, 'mithramycin': 1085.14544, 'itraconazole oral solution': 705.63342, 'Hydroxychloroquine sulfate': 0.0, 'mifepristone': 429.5937, 'metformin': 129.16364, 'acadesine': 258.23126, 'A-769662': 359.37794, 'imatinib': 493.60274, 'nilotinib': 529.51579, 'ponatinib': 532.55949, 'rebastinib': 553.586823, 'dasatinib': 488.00554, 'bosutinib': 530.4461

In [4]:
print("\nCleaning cell lines dataset...")
cell_lines = pd.read_csv("/content/cell_Line.csv")
print(cell_lines.isnull().sum())
if 'tag' in cell_lines.columns:
  missing = cell_lines['tag'].isnull().mean()
  if missing>0.5:
    cell_lines = cell_lines.drop('tag',axis=1)
cell_lines = cell_lines.dropna(subset=['cosmicId'])
print(cell_lines.isnull().sum())
drugcombs = pd.merge(drugcombs, cell_lines, left_on='Cell line', right_on='cellName', how='left')
drugcombs = drugcombs.drop(columns=['cellName'])
print(drugcombs.head())


Cleaning cell lines dataset...
cellName     0
cosmicId     8
tag         54
dtype: int64
cellName    0
cosmicId    0
dtype: int64
   ID Drug1    Drug2 Cell line    ZIP  Bliss  Loewe    HSA  Drug1_MW  \
0   1  5-FU  ABT-888     A2058   1.72   6.26  -2.75   5.54       0.0   
1   2  5-FU  ABT-888     A2058   5.88  12.33   3.33  11.61       0.0   
2   3  5-FU  ABT-888     A2058   3.59  11.66   2.65  10.94       0.0   
3   4  5-FU  ABT-888     A2058  -0.85   5.15  -3.86   4.43       0.0   
4   5  5-FU  AZD1775     A2058  12.29  15.77  10.40  18.66       0.0   

   Drug2_MW  cosmicId  
0       0.0  906792.0  
1       0.0  906792.0  
2       0.0  906792.0  
3       0.0  906792.0  
4       0.0  906792.0  


In [5]:
print("\nCleaning response data")
response_data = pd.read_csv("/content/drugcombs_response.csv", encoding='latin1')
#removing negative response values
response_data = response_data[response_data['Response']>=0]
print("Response data missing values:", response_data.isnull().sum())

response_features = response_data.groupby('BlockID').agg(
    response_mean = ('Response', 'mean'),
    response_min = ('Response', 'min'),
    response_max = ('Response', 'max')
).reset_index()

#merging the drugcombs dataset
drugcombs = pd.merge(drugcombs, response_features, left_on='ID', right_on='BlockID', how='left')
print(drugcombs.head())

print(drugcombs.isnull().sum())
drugcombs = drugcombs.dropna(subset=['cosmicId'])
drugcombs = drugcombs.dropna(subset=['BlockID', 'response_mean', 'response_min', 'response_max'])
print(drugcombs.isnull().sum())


Cleaning response data
Response data missing values: BlockID        0
Row            0
Col            0
DrugRow        0
DrugCol        0
ConcRow        0
ConcCol        0
Response       0
ConcRowUnit    0
ConcColUnit    0
source         0
dtype: int64
   ID Drug1    Drug2 Cell line    ZIP  Bliss  Loewe    HSA  Drug1_MW  \
0   1  5-FU  ABT-888     A2058   1.72   6.26  -2.75   5.54       0.0   
1   2  5-FU  ABT-888     A2058   5.88  12.33   3.33  11.61       0.0   
2   3  5-FU  ABT-888     A2058   3.59  11.66   2.65  10.94       0.0   
3   4  5-FU  ABT-888     A2058  -0.85   5.15  -3.86   4.43       0.0   
4   5  5-FU  AZD1775     A2058  12.29  15.77  10.40  18.66       0.0   

   Drug2_MW  cosmicId  BlockID  response_mean  response_min  response_max  
0       0.0  906792.0      1.0      85.222933     49.611875    112.350610  
1       0.0  906792.0      2.0      81.333453     17.988610    109.013513  
2       0.0  906792.0      3.0      81.764613     40.973380    101.993009  
3       0

In [6]:
print("\nCleaning gene expression dataset...")
gene_data = pd.read_csv("/content/Expression_(Short-read)_Public_26Q1_subsetted.csv")
gene_data = gene_data.groupby('cell_line_display_name').mean(numeric_only=True)
#removing non gene numerical metadata columns
metadata_cols = [c for c in gene_data.columns if 'lineage' in c.lower() or 'id' in c.lower()]
gene_features_only = gene_data.drop(columns=metadata_cols, errors='ignore')
#selecting 50 genes
selected_genes = gene_features_only.columns[:50]
gene_final = gene_features_only[selected_genes]

gene_data = gene_data.fillna(gene_final.mean())
gene_data = gene_data.drop(columns=['lineage_4'])
print(gene_data.head())


Cleaning gene expression dataset...


/tmp/ipykernel_1539/232562891.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  gene_data = pd.read_csv("/content/Expression_(Short-read)_Public_26Q1_subsetted.csv")


                            RBM5      RBM6  SLC25A13    CACNG3    ZNF263  \
cell_line_display_name                                                     
1156QE8                 4.214855  3.365992  5.115428  0.000000  3.526269   
127399                  5.684171  5.186893  5.131622  0.000000  4.431676   
143B                    5.677731  4.988889  4.411297  0.000000  4.399848   
1618K                   5.562944  5.876927  4.336176  0.184921  5.116793   
170MGBA                 5.427033  4.554712  4.666847  0.000000  4.478706   

                           CCL26     USH1C      AASS     TENM1     MASP2  ...  \
cell_line_display_name                                                    ...   
1156QE8                 0.746987  4.392798  4.988985  0.000000  0.051550  ...   
127399                  0.214137  0.059740  0.346435  0.000000  0.434584  ...   
143B                    0.000000  0.000000  3.505434  0.003150  0.309166  ...   
1618K                   4.139731  0.186866  6.206809  0.830071

In [7]:
gene_data = gene_data.reset_index().rename(columns = {'cell_line_display_name' : 'Cell line'})
drugcombs = pd.merge(drugcombs, gene_data, on='Cell line', how='inner')
print(drugcombs.head())

   ID Drug1    Drug2 Cell line    ZIP  Bliss  Loewe    HSA  Drug1_MW  \
0   1  5-FU  ABT-888     A2058   1.72   6.26  -2.75   5.54       0.0   
1   2  5-FU  ABT-888     A2058   5.88  12.33   3.33  11.61       0.0   
2   3  5-FU  ABT-888     A2058   3.59  11.66   2.65  10.94       0.0   
3   4  5-FU  ABT-888     A2058  -0.85   5.15  -3.86   4.43       0.0   
4   5  5-FU  AZD1775     A2058  12.29  15.77  10.40  18.66       0.0   

   Drug2_MW  ...      PJA2      MAFB    CRYZL1   PLEKHM1     CD302    AP5Z1  \
0       0.0  ...  4.853187  0.852875  4.866354  2.936227  3.430554  4.26552   
1       0.0  ...  4.853187  0.852875  4.866354  2.936227  3.430554  4.26552   
2       0.0  ...  4.853187  0.852875  4.866354  2.936227  3.430554  4.26552   
3       0.0  ...  4.853187  0.852875  4.866354  2.936227  3.430554  4.26552   
4       0.0  ...  4.853187  0.852875  4.866354  2.936227  3.430554  4.26552   

     CDK11B      TRIL     RBM8A      SCO2  
0  4.752888  0.005085  7.538872  4.902681  
1  4

In [8]:
#final merge
identifier_cols = ['ID','Drug1','Drug2','Cell line', 'cosmicId']
gene_cols = list(selected_genes)
target_col = ['ZIP','Bliss','Loewe','HSA']
final_cols = identifier_cols  + gene_cols + target_col
final_dataset = drugcombs[final_cols]
final_dataset.to_csv('final_dataset.csv', index=False)
print(f"Final dataset shape = {final_dataset.shape}")

Final dataset shape = (18068, 59)


In [9]:
df_summary = pd.read_csv('summary_gdsc_integrated.csv')
df_final = pd.read_csv('final_dataset.csv')

ic50_data = df_summary[['drug_row', 'drug_col', 'cell_line_name', 'ic50_row', 'ic50_col']].copy()
ic50_data.rename(columns={
    'drug_row': 'Drug1',
    'drug_col': 'Drug2',
    'cell_line_name': 'Cell line'
}, inplace=True)

ic50_data = ic50_data.drop_duplicates(subset=['Drug1', 'Drug2', 'Cell line'])

merged_dataset = pd.merge(df_final, ic50_data, on=['Drug1', 'Drug2', 'Cell line'], how='left')

merged_dataset.to_csv('merged_dataset_with_ic50.csv', index=False)

In [10]:

df_merged = pd.read_csv('/content/merged_dataset_with_ic50.csv')
df_filtered = df_merged[df_merged['ic50_row'].notna()]
df_filtered.to_csv('filtered_merged_dataset_with_ic50.csv', index=False)

In [12]:
df = pd.read_csv('/content/filtered_merged_dataset_with_ic50.csv')

# 2. Calculate Drug Response Score (DRS)
median_ic50_row = df['ic50_row'].median()
median_ic50_col = df['ic50_col'].median()

df['DRS_Drug1'] = -np.log2(df['ic50_row'] / median_ic50_row)
df['DRS_Drug2'] = -np.log2(df['ic50_col'] / median_ic50_col)
df['DRS_Combined'] = (df['DRS_Drug1'] + df['DRS_Drug2']) / 2

# 3. Calculate Drug Synergy (DS) Score
# Since Bliss, Loewe, HSA, and ZIP are already calculated in your dataset.
# A DS_Score > 0 generally indicates synergistic interaction.
df['DS_Score'] = df[['ZIP', 'Bliss', 'Loewe', 'HSA']].mean(axis=1)

# 4. Save the final dataset for your project
output_filename = 'final_project_dataset_with_DRS_DS.csv'
df.to_csv(output_filename, index=False)

print(f"Dataset successfully processed and saved as: {output_filename}")
print(df.shape)
print("\nSample of calculated scores:")
print(df[['Drug1', 'Drug2', 'DRS_Combined', 'DS_Score']].head())

Dataset successfully processed and saved as: final_project_dataset_with_DRS_DS.csv
(2272, 65)

Sample of calculated scores:
        Drug1       Drug2  DRS_Combined  DS_Score
0  BORTEZOMIB   DASATINIB      2.286606    4.9150
1  BORTEZOMIB   DASATINIB      2.286606   -2.3125
2  BORTEZOMIB   DASATINIB      2.286606    0.2625
3  BORTEZOMIB   DASATINIB      2.286606    2.6825
4  BORTEZOMIB  DINACICLIB      4.619368    1.6675
